# Chapter 15 — NLP 기반 SQLi/XSS 탐지 (실습)



In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q scikit-learn

In [ ]:
import re
import numpy as np
import pandas as pd
from urllib.parse import unquote
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
rng = np.random.default_rng(42)

In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

---
## Step 1. 합성 CSIC 데이터 생성

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 합성 CSIC 데이터 생성
# 만들 결과: normal_tpl, sqli_tpl, xss_tpl, borderline_tpl, WORDS, vary()
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 실무형 데이터셋: 템플릿을 랜덤 파라미터로 '고유화'해 train/test 중복 누수를 없애고,
# 공격 토큰을 품은 '정상 경계 사례'를 섞어 100% 분리(과완벽)를 막는다.
normal_tpl = ...
    ...
sqli_tpl = ...
    ...
xss_tpl = ...
    ...
# 공격 토큰을 품었지만 정상인 경계 사례 (오탐 FP 유발)
borderline_tpl = ...
    ...
WORDS = ...

def vary(tpl, k):
    out = ...
    for _ in range(k):
        s = ...
                       c = ...
                       w = ...
        sep = ...
        out.append(f"{s}{sep}_r = ...
    return ...

normal_payloads = ...
borderline_payloads = ...
sqli_payloads = ...
xss_payloads = ...

df = ...
    ...
}).drop_duplicates('payload').sample(frac = ...
df['label_bin'] = ...
# 라벨 노이즈 6% — 실제 데이터셋의 라벨링 오류로 완벽 분리(과완벽)를 막는다
_noise = ...
df.loc[_noise, 'label_bin'] = ...
...
```
</details>


---
## Step 2. 정규화

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 정규화
# 만들 결과: normalize(), new, s, df['payload_norm']
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def normalize(s: ...
    for _ in range(3):
        new = ...
        if new == s: ...
        s = ...
    s = ...
    s = ...
    return ...

df['payload_norm'] = ...
df[['payload', 'payload_norm']].sample(8, random_state = ...
```
</details>


---
## Step 3. TF-IDF char-ngram + LogReg

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. TF-IDF char-ngram + LogReg
# 만들 결과: pipe_lr, proba_lr
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
X_train, X_test, y_train, y_test = ...
    df['payload_norm'], df['label_bin'], test_size = ...

pipe_lr = ...
    ('tfidf', TfidfVectorizer(analyzer = ...
    ('clf',   LogisticRegression(max_iter = ...
...
proba_lr = ...
print(f'LR AUC = ...
...
```
</details>


---
## Step 4. RandomForest 비교

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. RandomForest 비교
# 만들 결과: pipe_rf, proba_rf
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
pipe_rf = ...
    ('tfidf', TfidfVectorizer(analyzer = ...
    ('clf',   RandomForestClassifier(n_estimators = ...
...
proba_rf = ...
print(f'RF AUC = ...
```
</details>


In [ ]:
# ROC 곡선 + 혼동행렬 — 완벽하지 않은(현실적인) 탐지기의 성능
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc, confusion_matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for name, pipe, color in [('LR', pipe_lr, '#1a5f7a'), ('RF', pipe_rf, '#e74c3c')]:
    pr = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, pr)
    ax1.plot(fpr, tpr, color=color, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
ax1.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
ax1.set_xlabel('거짓 양성률 (FPR)'); ax1.set_ylabel('참 양성률 (TPR)')
ax1.set_title('ROC 곡선'); ax1.legend(loc='lower right')
cm = confusion_matrix(y_test, pipe_rf.predict(X_test))
ax2.imshow(cm, cmap='Blues')
ax2.set_xticks([0, 1]); ax2.set_xticklabels(['정상', '공격'])
ax2.set_yticks([0, 1]); ax2.set_yticklabels(['정상', '공격'])
ax2.set_xlabel('예측'); ax2.set_ylabel('실제'); ax2.set_title('혼동 행렬 (RF)')
for (r, cc), v in np.ndenumerate(cm):
    ax2.text(cc, r, str(v), ha='center', va='center',
             color='white' if v > cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

---
## Step 5. 우회 페이로드 강건성 테스트

In [ ]:
# ── 실습 과제 ──
# 목표: Step 5. 우회 페이로드 강건성 테스트
# 만들 결과: evasion_examples, test, test['norm'], test['prob']
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
evasion_examples = ...
    ...
test = ...
test['norm'] = ...
test['prob'] = ...
...
```
</details>


---
## 정리
- char-ngram TF-IDF 가 보안 텍스트 분류의 표준 기준 모델.
- URL 디코딩 후 모델에 입력하면 인코딩 우회가 무력화.
- RF 가 LR 보다 약간 더 robust 한 경우 많음.